In [6]:
# Cell 1: import
import numpy as np
import pandas as pd
import math
import re
from collections import Counter


In [7]:
# Cell 2: Data
df = pd.read_csv(
    '../data/raw/MINDsmall_train/news.tsv',
    sep='\t', header=None,
    names=['id','category','subcategory','title',
           'abstract','url','title_entities','abstract_entities']
)
df['abstract'] = df['abstract'].fillna('')
df['text'] = df['title'] + ' ' + df['abstract']
print(df.shape)
df[['id','title','abstract', 'text']].head(5)

(51282, 9)


,id,title,abstract,text
0,N55528,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...","The Brands Queen Elizabeth, Prince Charles, an..."
1,N19639,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,50 Worst Habits For Belly Fat These seemingly ...
2,N61837,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,The Cost of Trump's Aid Freeze in the Trenches...
3,N53526,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",I Was An NBA Wife. Here's How It Affected My M...
4,N38324,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...","How to Get Rid of Skin Tags, According to a De..."


In [8]:
# Cell 3: text to tokens
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    return tokens

df["tokens"] = df["text"].apply(preprocess)
df[['title','abstract','tokens']].head(5)

,title,abstract,tokens
0,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...","[the, brands, queen, elizabeth, prince, charle..."
1,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,"[50, worst, habits, for, belly, fat, these, se..."
2,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,"[the, cost, of, trump, s, aid, freeze, in, the..."
3,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...","[i, was, an, nba, wife, here, s, how, it, affe..."
4,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...","[how, to, get, rid, of, skin, tags, according,..."


In [9]:
# cell4: tính df: là số lượng tài liệu chứa từ đó
df_counter = Counter()

for tokens in df["tokens"]:
    unique_tokens = set(tokens)
    
    for token in unique_tokens:
        df_counter[token] += 1

df_counter

df_df = pd.DataFrame({
    "word": list(df_counter.keys()),
    "df": list(df_counter.values())
})

df_df = df_df.sort_values("df", ascending=False).reset_index(drop=True)

df_df

,word,df
0,the,39330
1,to,30920
2,a,29509
3,in,28582
4,of,26355
...,...,...
54907,starves,1
54908,ballenger,1
54909,jerami,1
54910,ferrellgas,1


In [10]:
#cell 5
min_df = 1
max_df_ratio = 1.0

n_docs = len(df)

vocab = []

for word, df_value in df_counter.items():
    df_ratio = df_value / n_docs
    
    if df_value >= min_df and df_ratio <= max_df_ratio:
        vocab.append(word)

vocab = sorted(vocab)

print("Số document:", n_docs)
print("Số từ trong vocabulary:", len(vocab))

Số document: 51282
Số từ trong vocabulary: 54912


In [11]:
# Cell 6: Tạo word_to_index
word_to_index = {
    word: idx for idx, word in enumerate(vocab)
}

word_to_index

vocab_df = pd.DataFrame({
    "word": vocab,
    "word_id": [word_to_index[word] for word in vocab]
})

vocab_df

,word,word_id
0,0,0
1,00,1
2,000,2
3,000hp,3
4,000mah,4
...,...,...
54907,zykevious,54907
54908,zykov,54908
54909,zyuzin,54909
54910,zzzs,54910


In [ ]:
# Cell 7: Tính idf 
idf = {}

for word in vocab:
    df_value = df_counter[word]
    idf[word] = math.log((n_docs + 1) / (df_value + 1)) + 1

idf_df = pd.DataFrame({
    "word": vocab,
    "df": [df_counter[word] for word in vocab],
    "idf": [idf[word] for word in vocab]
})

idf_df = idf_df.sort_values("idf", ascending=False).reset_index(drop=True)

idf_df

,word,df,idf
0,kish,1,11.151967
1,lemarcus,1,11.151967
2,lelling,1,11.151967
3,lekraig,1,11.151967
4,leith,1,11.151967
...,...,...,...
54907,of,26355,1.665663
54908,in,28582,1.584547
54909,a,29509,1.552630
54910,to,30920,1.505924


In [13]:
# Cell 8: Tính tf cho từng document: Tần suất của mỗi từ trong document
tf_sparse = {}

for doc_idx, tokens in enumerate(df["tokens"]):
    tokens = [token for token in tokens if token in word_to_index]
    
    total_terms = len(tokens)
    term_counts = Counter(tokens)
    
    doc_tf = {}
    
    for word, count in term_counts.items():
        tf = count / total_terms
        word_id = word_to_index[word]
        doc_tf[word_id] = tf
    
    tf_sparse[doc_idx] = doc_tf

tf_sparse[0]

index_to_word = {
    idx: word for word, idx in word_to_index.items()
}

tf_doc0 = []

for word_id, tf_value in tf_sparse[0].items():
    tf_doc0.append({
        "word_id": word_id,
        "word": index_to_word[word_id],
        "tf": tf_value
    })

pd.DataFrame(tf_doc0)

,word_id,word,tf
0,48989,the,0.125000
1,7620,brands,0.041667
2,39311,queen,0.041667
3,16735,elizabeth,0.041667
4,38527,prince,0.083333
5,9913,charles,0.041667
6,3567,and,0.083333
7,37101,philip,0.041667
8,47917,swear,0.041667
9,8536,by,0.041667


In [22]:
# Cell 9: Tính tf-idf sparse
tfidf_sparse = {}

for doc_idx, tokens in enumerate(df["tokens"]):
    tokens = [token for token in tokens if token in word_to_index]
    
    total_terms = len(tokens)
    
    if total_terms == 0:
        tfidf_sparse[doc_idx] = {}
        continue
    
    term_counts = Counter(tokens)
    doc_vector = {}
    
    for word, count in term_counts.items():
        tf = count / total_terms
        tfidf_value = tf * idf[word]
        
        word_id = word_to_index[word]
        doc_vector[word_id] = tfidf_value
    
    tfidf_sparse[doc_idx] = doc_vector

def show_tfidf_doc(doc_idx):
    rows = []
    
    for word_id, value in tfidf_sparse[doc_idx].items():
        word = index_to_word[word_id]
        
        rows.append({
            "doc_id": doc_idx,
            "word_id": word_id,
            "word": word,
            "tfidf": value
        })
    
    result = pd.DataFrame(rows)
    result = result.sort_values("tfidf", ascending=False).reset_index(drop=True)
    
    return result

selected_docs = [5]

for doc_idx in selected_docs:
    print("Document", doc_idx)
    print(df.loc[doc_idx, "text"])
    display(show_tfidf_doc(doc_idx))
    print("-" * 60)

Document 5
Should NFL be able to fine players for criticizing officiating? Several fines came down against NFL players for criticizing officiating this week. It's a very bad look for the league.


,doc_id,word_id,word,tfidf
0,5,12739,criticizing,0.551566
1,5,34871,officiating,0.520060
2,5,37575,players,0.338140
3,5,33909,nfl,0.308908
4,5,18931,fines,0.259305
5,5,18925,fine,0.225836
6,5,19548,for,0.194021
7,5,2099,able,0.192226
8,5,5162,bad,0.189644
9,5,52148,very,0.175427


------------------------------------------------------------


In [16]:
# cell 10: Tính cosine similarity
def sparse_dot(vec_a, vec_b):
    result = 0.0
    
    if len(vec_a) > len(vec_b):
        vec_a, vec_b = vec_b, vec_a
    
    for idx, value in vec_a.items():
        if idx in vec_b:
            result += value * vec_b[idx]
    
    return result


def sparse_norm(vec):
    total = 0.0
    
    for value in vec.values():
        total += value ** 2
    
    return math.sqrt(total)


def sparse_cosine_similarity(vec_a, vec_b):
    norm_a = sparse_norm(vec_a)
    norm_b = sparse_norm(vec_b)
    
    if norm_a == 0 or norm_b == 0:
        return 0.0
    
    return sparse_dot(vec_a, vec_b) / (norm_a * norm_b)

In [28]:
#cell 11: Tìm docs giống
def find_similar_docs(query_doc_idx, top_k=3):
    query_vec = tfidf_sparse[query_doc_idx]
    scores = []
    
    for doc_idx, doc_vec in tfidf_sparse.items():
        if doc_idx == query_doc_idx:
            continue
        
        score = sparse_cosine_similarity(query_vec, doc_vec)
        
        scores.append({
            "doc_id": doc_idx,
            "text": df.loc[doc_idx, "text"],
            "similarity": score
        })
    
    result = pd.DataFrame(scores)
    result = result.sort_values("similarity", ascending=False).reset_index(drop=True)
    
    return result.head(top_k)

find_similar_docs(100, top_k=5)

,doc_id,text,similarity
0,13712,When Celebs Dressed Up as Other Celebs for Hal...,0.352443
1,31508,"Who's Due Next? Kimberly, Malika, Hilaria and ...",0.321766
2,16,Stars who got fired from major projects Take a...,0.275075
3,13382,50 things every woman over 50 should know abou...,0.263745
4,328,"Now That I'm Over 50, I'm Finally Sucking It U...",0.247311
